# 2.3. Предварительный анализ, очистка, аннотирование и слияние данных

Цель — подготовить воспроизводимую объединённую витрину данных на уровне **Taxi Zone × дата × час**.

Основой служат поездки NYC Yellow Taxi за январь 2025 года. ДТП и обращения 311 присоединяются после пространственного соединения их координат с полигонами Taxi Zone. Пресс-релизы NYC DOT аннотируются и векторизуются как неструктурированный текст; они добавляются только как городской/районный временной контекст.

In [1]:
import sys
!{sys.executable} -m pip install scikit-learn==1.5.2

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Ноутбук можно открыть из папки проекта или из notebooks/.
start_dir = Path.cwd().resolve()
PROJECT_DIR = next(folder for folder in [start_dir, *start_dir.parents] if (folder / 'data').exists())
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.pipeline_2_3 import overview, run_pipeline

pd.set_option('display.max_columns', 100)
print(f'Папка проекта: {PROJECT_DIR}')

Папка проекта: C:\Users\User\Documents\3 курс\ТОБД\GeoData


## План очистки и ограничения

Для поездок исключаются записи без времени, с недопустимыми идентификаторами зон, отрицательными расстоянием/стоимостью и длительностью вне диапазона от 1 секунды до 3 часов. Каждое исключение фиксируется в журнале. Это не исправление исходных значений «на глаз», а прозрачная фильтрация наблюдений, которые не описывают обычную поездку.

ДТП и 311 — точечные данные: их широта и долгота автоматически сопоставляются с границами Taxi Zone. Для 311 берутся только обращения, связанные со street/traffic/road/signal/sidewalk/parking/highway/bridge/construction/transport. XML-файл в проекте является выборкой, поэтому показатель `service311_sample_count` не интерпретируется как полное число всех обращений Нью-Йорка.

In [3]:
# Полный конвейер: загрузка, профиль, очистка, пространственное соединение,
# аннотирование текста, TF-IDF-векторизация, агрегация и сохранение результатов.
# Выполнение может занять несколько минут: основной файл содержит миллионы поездок.
result = run_pipeline(PROJECT_DIR, month='2025-01')

print('Объединённая витрина создана:', result['fact'].shape)
print('Файлы сохранены в:', PROJECT_DIR / 'data' / 'processed')

Объединённая витрина создана: (95173, 19)
Файлы сохранены в: C:\Users\User\Documents\3 курс\ТОБД\GeoData\data\processed


## Количественные характеристики, типы и пропуски

Следующая ячейка выводит размер наборов, типы столбцов, число и долю пропусков, число уникальных значений и пример значения. Это одновременно документирует структуру данных до очистки и позволяет обосновать правила обработки.

In [4]:
raw_frames = {
    'Yellow Taxi (Parquet)': result['taxi_raw'],
    'NYPD Collisions (JSON)': result['collisions_raw'],
    'NYC 311 (XML)': result['service_raw'],
    'NYC DOT text (HTML)': result['dot_text'],
}

for name, frame in raw_frames.items():
    print(f'\n{name}: {frame.shape[0]:,} строк, {frame.shape[1]} столбцов')
    display(overview(frame, name))


Yellow Taxi (Parquet): 3,475,226 строк, 20 столбцов


,source,column,dtype,missing_count,missing_share,unique_count,example_value
0,Yellow Taxi (Parquet),VendorID,int32,0,0.00,4,1
1,Yellow Taxi (Parquet),tpep_pickup_datetime,datetime64[us],0,0.00,1672077,2025-01-01 00:18:38
2,Yellow Taxi (Parquet),tpep_dropoff_datetime,datetime64[us],0,0.00,1671993,2025-01-01 00:26:59
3,Yellow Taxi (Parquet),passenger_count,float64,540149,15.54,10,1.0
4,Yellow Taxi (Parquet),trip_distance,float64,0,0.00,4545,1.6
5,Yellow Taxi (Parquet),RatecodeID,float64,540149,15.54,7,1.0
6,Yellow Taxi (Parquet),store_and_fwd_flag,object,540149,15.54,2,N
7,Yellow Taxi (Parquet),PULocationID,int32,0,0.00,261,229
8,Yellow Taxi (Parquet),DOLocationID,int32,0,0.00,260,237
9,Yellow Taxi (Parquet),payment_type,int64,0,0.00,6,1



NYPD Collisions (JSON): 6,482 строк, 29 столбцов


,source,column,dtype,missing_count,missing_share,unique_count,example_value
0,NYPD Collisions (JSON),crash_date,object,0,0.00,31,2025-01-01T00:00:00.000
1,NYPD Collisions (JSON),crash_time,object,0,0.00,1253,11:20
2,NYPD Collisions (JSON),borough,object,1338,20.64,5,BROOKLYN
3,NYPD Collisions (JSON),zip_code,object,1342,20.70,182,11213
4,NYPD Collisions (JSON),latitude,object,79,1.22,5342,40.66658
5,NYPD Collisions (JSON),longitude,object,79,1.22,5137,-73.939674
6,NYPD Collisions (JSON),location,object,79,1.22,5545,"{'latitude': '40.66658', 'longitude': '-73.939..."
7,NYPD Collisions (JSON),on_street_name,object,1966,30.33,1515,ALBANY AVE
8,NYPD Collisions (JSON),off_street_name,object,2660,41.04,1706,CARROLL ST
9,NYPD Collisions (JSON),number_of_persons_injured,object,0,0.00,10,0



NYC 311 (XML): 1,000 строк, 42 столбцов


,source,column,dtype,missing_count,missing_share,unique_count,example_value
0,NYC 311 (XML),row,float64,1000,100.0,0,None
1,NYC 311 (XML),unique_key,object,0,0.0,1000,63577994
2,NYC 311 (XML),created_date,object,0,0.0,898,2025-01-01T00:00:12
3,NYC 311 (XML),closed_date,object,1,0.1,876,2025-01-02T17:05:54
4,NYC 311 (XML),agency,object,0,0.0,10,NYPD
5,NYC 311 (XML),agency_name,object,0,0.0,10,New York City Police Department
6,NYC 311 (XML),complaint_type,object,0,0.0,30,Noise - Residential
7,NYC 311 (XML),descriptor,object,2,0.2,42,Loud Music/Party
8,NYC 311 (XML),location_type,object,24,2.4,14,Residential Building/House
9,NYC 311 (XML),incident_zip,object,5,0.5,154,10466



NYC DOT text (HTML): 10 строк, 8 столбцов


,source,column,dtype,missing_count,missing_share,unique_count,example_value
0,NYC DOT text (HTML),title,object,0,0.0,8,NYC DOT Promotes Easy and Affordable Outdoor\r...
1,NYC DOT text (HTML),url,object,0,0.0,10,https://www.nyc.gov/html/dot/html/pr2025/easy-...
2,NYC DOT text (HTML),article_text,object,0,0.0,10,NYC DOT Promotes Easy and Affordable Outdoor D...
3,NYC DOT text (HTML),published_date,datetime64[ns],0,0.0,9,2025-01-30 00:00:00
4,NYC DOT text (HTML),borough_mentioned,object,0,0.0,4,Staten Island
5,NYC DOT text (HTML),topic,object,0,0.0,1,construction
6,NYC DOT text (HTML),text_length,int64,0,0.0,10,4831
7,NYC DOT text (HTML),transport_word_count,int64,0,0.0,7,22


## Физический смысл и аналитическая роль используемых атрибутов

Ниже перечислены атрибуты, непосредственно входящие в очистку, агрегацию или слияние. Полная классификация **всех исходных столбцов** по расширенным типам шкал дополнительно сохранена в `reports/attribute_scales_2_2.md`. Технические идентификаторы не усредняются: они используются только для соединения, группировки или подсчёта объектов.

In [5]:
attribute_catalog = pd.DataFrame([
    ['Yellow Taxi', 'PULocationID, DOLocationID', 'ID зон посадки/высадки', 'ключ соединения и маршрута', 'номинальная ID', 'позволяют выявлять спрос и частые направления'],
    ['Yellow Taxi', 'tpep_pickup_datetime, tpep_dropoff_datetime', 'время начала и конца поездки', 'временная ось', 'интервальная', 'час и дата — основа пространственно-временного паттерна'],
    ['Yellow Taxi', 'trip_distance', 'пройденное расстояние в милях', 'числовой показатель', 'отношений', 'характеризует дальность поездок'],
    ['Yellow Taxi', 'passenger_count', 'заявленное число пассажиров', 'числовой показатель', 'абсолютная', 'приближённо характеризует объём перевозки'],
    ['Yellow Taxi', 'fare_amount, total_amount', 'стоимость без доплат / итоговая стоимость', 'числовой показатель', 'отношений', 'характеризуют экономический аспект поездки'],
    ['Taxi Zones', 'LocationID', 'официальный идентификатор зоны', 'географический ключ', 'номинальная ID', 'связывает поездки и точки событий'],
    ['Taxi Zones', 'zone, borough, geometry', 'название зоны, боро и полигон границы', 'географическое измерение', 'категориальная / контентная', 'дают читаемую территориальную интерпретацию'],
    ['NYPD Collisions', 'crash_date, crash_time', 'дата и время ДТП', 'временной ключ', 'интервальная', 'позволяют агрегировать ДТП по часу'],
    ['NYPD Collisions', 'latitude, longitude', 'координаты ДТП', 'пространственный ключ', 'интервальная', 'назначаются Taxi Zone через полигон'],
    ['NYPD Collisions', 'collision_id', 'идентификатор ДТП', 'идентификатор', 'номинальная ID', 'используется для подсчёта происшествий'],
    ['NYPD Collisions', 'number_of_persons_injured, number_of_persons_killed', 'число пострадавших и погибших', 'числовой показатель', 'абсолютная', 'характеризуют тяжесть дорожной обстановки'],
    ['NYC 311', 'created_date, closed_date', 'время создания и закрытия обращения', 'временной ключ', 'интервальная', 'позволяют считать длительность обработки'],
    ['NYC 311', 'complaint_type, descriptor, agency, status', 'тип, описание, ведомство и статус обращения', 'контекстный признак', 'категориальная', 'выделяют транспортно значимые обращения'],
    ['NYC 311', 'latitude, longitude', 'координаты обращения', 'пространственный ключ', 'интервальная', 'назначаются Taxi Zone через полигон'],
    ['NYC DOT text', 'article_text, title, url', 'текст, заголовок и ссылка на пресс-релиз', 'неструктурированный контент', 'контентная', 'описывает транспортный контекст города'],
    ['NYC DOT text', 'published_date, borough_mentioned, topic', 'дата и автоматически выделенные категории', 'аннотация текста', 'интервальная / категориальная', 'используются для временного и районного контекста'],
    ['NYC DOT text', 'text_length, transport_word_count', 'длина текста и число транспортных слов', 'векторизованный признак', 'абсолютная', 'характеризуют насыщенность транспортной темой'],
], columns=['source', 'attribute', 'physical_meaning', 'analytical_role', 'scale', 'hypothesis_relevance'])
display(attribute_catalog)

,source,attribute,physical_meaning,analytical_role,scale,hypothesis_relevance
0,Yellow Taxi,"PULocationID, DOLocationID",ID зон посадки/высадки,ключ соединения и маршрута,номинальная ID,позволяют выявлять спрос и частые направления
1,Yellow Taxi,"tpep_pickup_datetime, tpep_dropoff_datetime",время начала и конца поездки,временная ось,интервальная,час и дата — основа пространственно-временного...
2,Yellow Taxi,trip_distance,пройденное расстояние в милях,числовой показатель,отношений,характеризует дальность поездок
3,Yellow Taxi,passenger_count,заявленное число пассажиров,числовой показатель,абсолютная,приближённо характеризует объём перевозки
4,Yellow Taxi,"fare_amount, total_amount",стоимость без доплат / итоговая стоимость,числовой показатель,отношений,характеризуют экономический аспект поездки
5,Taxi Zones,LocationID,официальный идентификатор зоны,географический ключ,номинальная ID,связывает поездки и точки событий
6,Taxi Zones,"zone, borough, geometry","название зоны, боро и полигон границы",географическое измерение,категориальная / контентная,дают читаемую территориальную интерпретацию
7,NYPD Collisions,"crash_date, crash_time",дата и время ДТП,временной ключ,интервальная,позволяют агрегировать ДТП по часу
8,NYPD Collisions,"latitude, longitude",координаты ДТП,пространственный ключ,интервальная,назначаются Taxi Zone через полигон
9,NYPD Collisions,collision_id,идентификатор ДТП,идентификатор,номинальная ID,используется для подсчёта происшествий


## Результаты очистки и примеры наблюдений

Журнал показывает число строк, исключённых по каждому последовательному правилу. Затем выводятся несколько реальных строк уже очищенных наборов: так можно увидеть структуру данных, а не только сводные показатели.

In [6]:
print('Журнал очистки Yellow Taxi')
display(result['taxi_audit'])

print('Примеры очищенных поездок')
display(result['taxi'][['tpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance', 'trip_duration_min', 'total_amount']].head(5))

print('Примеры ДТП после назначения Taxi Zone')
display(result['collisions'][['crash_datetime', 'LocationID', 'zone_borough', 'number_of_persons_injured', 'number_of_persons_killed']].head(5))

print('Примеры релевантных обращений 311 после назначения Taxi Zone')
display(result['service'][['created_datetime', 'complaint_type', 'LocationID', 'zone_borough', 'resolution_hours']].head(5))

Журнал очистки Yellow Taxi


,rule,excluded_rows
0,есть время начала и окончания,0
1,корректные идентификаторы зон,28357
2,неотрицательная дистанция,0
3,длительность от 1 секунды до 3 часов,2790
4,неотрицательная итоговая стоимость,62304
5,время начала попадает в анализируемый месяц,22


Примеры очищенных поездок


,tpep_pickup_datetime,PULocationID,DOLocationID,trip_distance,trip_duration_min,total_amount
0,2025-01-01 00:18:38,229,237,1.60,8.350000,18.00
1,2025-01-01 00:32:40,236,237,0.50,2.550000,12.12
2,2025-01-01 00:44:04,141,141,0.60,1.950000,12.10
3,2025-01-01 00:14:27,244,244,0.52,5.566667,9.70
4,2025-01-01 00:21:34,244,116,0.66,3.533333,8.30


Примеры ДТП после назначения Taxi Zone


,crash_datetime,LocationID,zone_borough,number_of_persons_injured,number_of_persons_killed
0,2025-01-01 11:20:00,62.0,Brooklyn,0,0
1,2025-01-01 20:23:00,210.0,Brooklyn,1,0
2,2025-01-01 21:45:00,172.0,Staten Island,4,0
3,2025-01-01 09:25:00,204.0,Staten Island,0,0
4,2025-01-01 03:00:00,257.0,Brooklyn,3,0


Примеры релевантных обращений 311 после назначения Taxi Zone


,created_datetime,complaint_type,LocationID,zone_borough,resolution_hours
11,2025-01-01 00:01:06,Noise - Street/Sidewalk,74,Manhattan,0.299444
12,2025-01-01 00:01:24,Noise - Street/Sidewalk,74,Manhattan,0.703056
14,2025-01-01 00:01:42,Illegal Parking,189,Brooklyn,0.714722
17,2025-01-01 00:02:00,Traffic Signal Condition,226,Queens,0.000000
22,2025-01-01 00:02:24,Noise - Street/Sidewalk,75,Manhattan,0.288889


## Аннотирование и векторизация неструктурированного текста

Для каждого пресс-релиза автоматически создаются аннотации `topic` и `borough_mentioned`, извлекается дата, считаются длина текста и число транспортных слов. Затем `TfidfVectorizer` превращает текст каждого документа в числовой вектор TF–IDF. Значение TF–IDF выше для слов и фраз, характерных для конкретного документа, и ниже для часто встречающихся во всех материалах слов.

In [7]:
print('Аннотированные тексты NYC DOT')
display(result['dot_text'][['published_date', 'borough_mentioned', 'topic', 'title', 'text_length', 'transport_word_count']])

print('Первые признаки TF-IDF: строки — пресс-релизы, столбцы — слова/фразы')
display(result['text_vectors'].head())

print('Словарь признаков TF-IDF')
display(result['vocabulary'].head(20))

Аннотированные тексты NYC DOT


,published_date,borough_mentioned,topic,title,text_length,transport_word_count
0,2025-01-30,Staten Island,construction,NYC DOT Promotes Easy and Affordable Outdoor\r...,4831,22
1,2025-01-27,Manhattan,construction,NYC DOT Opens Applications for E-Bike Trade-In...,15336,78
2,2025-01-23,Staten Island,construction,NYC DOT’s Speed Cameras Achieved a 94 Percent ...,18370,57
3,2025-01-21,Staten Island,construction,Suspension of Alternate Side Parking\r\n ...,1635,24
4,2025-01-21,Manhattan,construction,Nights on Broadway : NYC DOT Releases\r\n ...,5921,31
5,2025-01-20,Staten Island,construction,Suspension of Alternate Side Parking\r\n ...,1633,24
6,2025-01-16,Manhattan,construction,New Episode of NYC DOT’s Curb Enthusiasm\r\n ...,6604,24
7,2025-01-15,Bronx,construction,Vision Zero: New Report Shows Street Redesigns...,12465,83
8,2025-01-14,Brooklyn,construction,NYC DOT Authorizes On-Street 'Microhub Zones' ...,10752,58
9,2025-01-10,Staten Island,construction,Suspension of Alternate Side Parking\r\n ...,1636,24


Первые признаки TF-IDF: строки — пресс-релизы, столбцы — слова/фразы


,document_id,accessibility,batteries,bike,bikes,broadway,camera,cameras,city,commissioner,contact,data,delivery,delivery workers,dining,dot,fatalities,locations,mobility,neighborhoods,new,new york,nyc,nyc dot,open,parking,pedestrian,plan,press,program,public,report,safe,safer,safety,said,speed,speed camera,speed cameras,speeding,street,streets,traffic,transportation,vision,vision zero,work,workers,york,york city,zero
0,0,0.084741,0.000000,0.105926,0.042371,0.000000,0.000000,0.021185,0.148297,0.055864,0.063556,0.021185,0.000000,0.000000,0.593187,0.296594,0.000000,0.000000,0.042371,0.0,0.127112,0.084741,0.593187,0.211853,0.084741,0.127112,0.042371,0.042371,0.063556,0.153803,0.000000,0.000000,0.021185,0.000000,0.063556,0.027932,0.000000,0.000000,0.000000,0.000000,0.063556,0.084741,0.063556,0.027932,0.021185,0.021185,0.021185,0.000000,0.084741,0.055864,0.021185
1,1,0.018215,0.394144,0.255015,0.121436,0.000000,0.000000,0.006072,0.151795,0.048032,0.018215,0.012144,0.341992,0.394144,0.006072,0.139651,0.000000,0.012214,0.030359,0.0,0.182154,0.121436,0.157866,0.085005,0.030359,0.036431,0.012144,0.030359,0.018215,0.290933,0.156034,0.000000,0.170010,0.097732,0.109292,0.128086,0.000000,0.000000,0.000000,0.000000,0.042503,0.066790,0.018215,0.048032,0.012144,0.006072,0.042503,0.390901,0.121436,0.088059,0.006072
2,2,0.011532,0.000000,0.026909,0.007688,0.000000,0.374304,0.134544,0.130700,0.030410,0.011532,0.038441,0.000000,0.000000,0.003844,0.084570,0.106064,0.131458,0.007688,0.0,0.149920,0.092258,0.096103,0.049973,0.015376,0.023065,0.007688,0.011532,0.011532,0.217682,0.012348,0.154656,0.019221,0.075625,0.065350,0.091229,0.644635,0.301523,0.239139,0.238644,0.030753,0.080726,0.061506,0.030410,0.034597,0.034597,0.023065,0.008839,0.092258,0.076024,0.038441
3,3,0.149256,0.000000,0.248759,0.099504,0.000000,0.000000,0.049752,0.099504,0.000000,0.149256,0.049752,0.000000,0.000000,0.049752,0.348263,0.000000,0.000000,0.099504,0.0,0.049752,0.049752,0.547270,0.149256,0.099504,0.497519,0.099504,0.099504,0.149256,0.000000,0.000000,0.000000,0.049752,0.000000,0.099504,0.000000,0.000000,0.000000,0.000000,0.000000,0.149256,0.199007,0.149256,0.000000,0.049752,0.049752,0.049752,0.000000,0.049752,0.000000,0.049752
4,4,0.043277,0.000000,0.072128,0.028851,0.667431,0.000000,0.014426,0.274088,0.057059,0.043277,0.014426,0.000000,0.000000,0.014426,0.259662,0.000000,0.000000,0.028851,0.0,0.245237,0.230811,0.302940,0.187534,0.028851,0.100980,0.028851,0.043277,0.043277,0.000000,0.023170,0.000000,0.014426,0.000000,0.028851,0.038039,0.000000,0.000000,0.000000,0.000000,0.187534,0.100980,0.043277,0.019020,0.028851,0.014426,0.014426,0.000000,0.230811,0.190196,0.014426


Словарь признаков TF-IDF


,term,idf
0,accessibility,1.0
21,nyc,1.0
22,nyc dot,1.0
23,open,1.0
25,pedestrian,1.0
26,plan,1.0
27,press,1.0
31,safe,1.0
33,safety,1.0
39,street,1.0


## Слияние и структура хранения

| Источник | Приведение к общему ключу | Способ присоединения |
|---|---|---|
| Yellow Taxi | `PULocationID`, дата, час | формирует основу витрины |
| Taxi Zones | `LocationID`, полигон, borough | соединение по ID зоны |
| ДТП | координаты → полигон Taxi Zone; дата, час | `Taxi Zone + дата + час` |
| NYC 311 | координаты → полигон Taxi Zone; дата, час | `Taxi Zone + дата + час` |
| NYC DOT text | borough и дата из текста | `borough + дата`; контекст распространяется на часы дня |

Выбрана **факт-таблица в Parquet** `fact_taxi_zone_hour_2025-01.parquet`. Parquet — колоночный формат: он компактнее CSV, быстрее читает отдельные поля и подходит для миллионов строк. Taxi Zone, дата и час — логический составной ключ. Геометрию зон выгоднее хранить отдельно как небольшое измерение Shapefile/GeoDataFrame, а не дублировать в каждой строке факт-таблицы.

In [8]:
fact = result['fact']
print(f'Витрина Taxi Zone × дата × час: {fact.shape[0]:,} строк, {fact.shape[1]} столбцов')
print('Пропуски в итоговой витрине:')
display(fact.isna().sum().sort_values(ascending=False).to_frame('missing_count').head(15))

print('Примеры объединённых данных:')
display(fact.head(10))

print('Проверка уникальности составного ключа:')
key_columns = ['taxi_zone_id', 'event_date', 'event_hour']
print('Повторяющихся ключей:', int(fact.duplicated(key_columns).sum()))

processed_path = PROJECT_DIR / 'data' / 'processed' / 'fact_taxi_zone_hour_2025-01.parquet'
print('Файл объединённой витрины:', processed_path)

Витрина Taxi Zone × дата × час: 95,173 строк, 19 столбцов
Пропуски в итоговой витрине:


,missing_count
resolution_hours_median,95087
taxi_zone_id,0
zone,0
dot_release_count,0
service311_sample_count,0
persons_killed_sum,0
persons_injured_sum,0
collision_count,0
borough,0
total_amount_sum,0


Примеры объединённых данных:


,taxi_zone_id,event_date,event_hour,trip_count,passenger_count_sum,trip_distance_mean,trip_duration_min_mean,fare_amount_mean,total_amount_mean,total_amount_sum,zone,borough,collision_count,persons_injured_sum,persons_killed_sum,service311_sample_count,resolution_hours_median,dot_release_count,dot_transport_word_count
0,1,2025-01-01,6,1,2.0,0.100000,0.266667,99.000000,115.00,115.00,Newark Airport,EWR,0,0,0,0,NaN,0,0
1,1,2025-01-01,9,4,5.0,0.000000,11.762500,115.140000,132.51,530.04,Newark Airport,EWR,0,0,0,0,NaN,0,0
2,1,2025-01-01,12,2,2.0,0.000000,0.075000,87.500000,106.20,212.40,Newark Airport,EWR,0,0,0,0,NaN,0,0
3,1,2025-01-01,13,3,6.0,1.333333,3.627778,48.333333,56.40,169.20,Newark Airport,EWR,0,0,0,0,NaN,0,0
4,1,2025-01-01,14,1,4.0,0.000000,0.133333,120.000000,121.01,121.01,Newark Airport,EWR,0,0,0,0,NaN,0,0
5,1,2025-01-01,15,2,2.0,0.000000,0.375000,24.000000,25.00,50.00,Newark Airport,EWR,0,0,0,0,NaN,0,0
6,1,2025-01-01,16,2,2.0,0.015000,1.450000,102.500000,113.60,227.20,Newark Airport,EWR,0,0,0,0,NaN,0,0
7,1,2025-01-01,17,1,1.0,0.000000,0.166667,120.000000,121.00,121.00,Newark Airport,EWR,0,0,0,0,NaN,0,0
8,1,2025-01-02,5,1,1.0,0.000000,0.483333,119.000000,144.00,144.00,Newark Airport,EWR,0,0,0,0,NaN,0,0
9,1,2025-01-02,7,1,2.0,0.000000,0.233333,94.000000,118.75,118.75,Newark Airport,EWR,0,0,0,0,NaN,0,0


Проверка уникальности составного ключа:
Повторяющихся ключей: 0
Файл объединённой витрины: C:\Users\User\Documents\3 курс\ТОБД\GeoData\data\processed\fact_taxi_zone_hour_2025-01.parquet


## Обновление данных

Вся логика вынесена также в `src/pipeline_2_3.py`. Для повторного формирования витрины после загрузки нового файла поездок используйте из корня проекта:

```powershell
python src/pipeline_2_3.py --month 2025-01
```

Скрипт повторно получает JSON и HTML, использует локальный XML и Parquet, применяет те же правила очистки, обновляет промежуточные профили, TF–IDF-векторы и итоговую Parquet-витрину. Поэтому процедура не зависит от ручного копирования результатов между таблицами.